<a href="https://colab.research.google.com/github/dharvi120/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [33]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [34]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Baseline Rule

My baseline rule prioritizes content pages for refresh using simple, observable search performance signals instead of a machine learning model.

A page receives a higher refresh score if it:

- Has not been updated recently.
- Has a below-median click-through rate (CTR).
- Has high search visibility (high impressions).
- Has a relatively poor average search position.

This rule is transparent, easy to interpret, and serves as the baseline that the Week 5 machine learning model should outperform.

### Reason Codes

- STALE_CONTENT – The page has not been updated for a long time.
- LOW_CTR – The page has a lower-than-median click-through rate.
- HIGH_VISIBILITY – The page receives many impressions and has refresh potential.
- POSITION_SLIPPING – The page has a relatively poor average search position.

In [36]:
import pandas as pd
import numpy as np

# Rule thresholds
CTR_THRESHOLD = df["ctr"].median()
IMP_THRESHOLD = df["impressions_90d"].median()
STALE_DAYS = 180
POSITION_THRESHOLD = 20

print("CTR Threshold:", CTR_THRESHOLD)
print("Impression Threshold:", IMP_THRESHOLD)

CTR Threshold: 0.07
Impression Threshold: 731.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [37]:
import os

df["baseline_score"] = 0

df.loc[df["days_since_last_update"] > STALE_DAYS, "baseline_score"] += 40
df.loc[df["ctr"] < CTR_THRESHOLD, "baseline_score"] += 30
df.loc[df["impressions_90d"] > IMP_THRESHOLD, "baseline_score"] += 20
df.loc[df["avg_position"] > POSITION_THRESHOLD, "baseline_score"] += 10

def reason_code(row):
    if row["days_since_last_update"] > STALE_DAYS:
        return "STALE_CONTENT"
    elif row["ctr"] < CTR_THRESHOLD:
        return "LOW_CTR"
    elif row["avg_position"] > POSITION_THRESHOLD:
        return "POSITION_SLIPPING"
    else:
        return "HIGH_VISIBILITY"

df["reason_code"] = df.apply(reason_code, axis=1)

df["action"] = np.where(
    df["baseline_score"] >= 60,
    "Refresh",
    "Monitor"
)

ranked = df.sort_values(
    by="baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")
ranked.head(10)

CSV written successfully.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.0,40.00,0.0,good,page_3_5,down,-89.1,100,STALE_CONTENT,Refresh
698,content_b16bd7307b39,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4329.0,27844.0,...,0.0,25.00,0.0,good,page_3_5,down,-69.7,100,STALE_CONTENT,Refresh
11494,content_d34c89fad803,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,4285.0,30371.0,...,0.0,33.33,0.0,low,page_3_5,up,35.0,80,STALE_CONTENT,Refresh
22860,content_ab18b5811c02,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,0.0,0.00,0.0,low,page_3_5,down,-100.0,80,STALE_CONTENT,Refresh
505,content_bfa3d6688324,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,4855.0,34332.0,...,0.0,66.67,0.0,low,page_3_5,down,-50.0,80,STALE_CONTENT,Refresh
7509,content_7a888d3d99c8,client_19581e27de,90.0,0.46,MEDIUM,0.72,keyword article,transactional,NaN,NaN,...,0.0,0.00,0.0,low,deep,down,-100.0,80,STALE_CONTENT,Refresh
18652,content_0173fb0dc986,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3584.0,25914.0,...,0.0,66.67,0.0,low,page_3_5,down,-92.1,80,STALE_CONTENT,Refresh
26249,content_dd413158df3c,client_19581e27de,20.0,1.00,HIGH,1.64,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,low,page_3_5,stable,-16.7,80,STALE_CONTENT,Refresh
3507,content_074ba6ead17b,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3994.0,27901.0,...,0.0,50.00,0.0,moderate,page_3_5,down,-36.5,80,STALE_CONTENT,Refresh
8860,content_adfc46f3f033,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,424.0,3143.0,...,0.0,100.00,0.0,low,page_3_5,down,-60.0,80,STALE_CONTENT,Refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [41]:
top20 = ranked.head(20)

for i, row in top20.iterrows():

    print("="*60)

    print("Content ID:", row["content_id"])

    print("Action:", row["action"])

    print("Reason Code:", row["reason_code"])

    print("Confidence: Medium")

    print("What would make it wrong?")

    print("- Traffic may be seasonal.")
    print("- Recent Google algorithm changes.")
    print("- Content may have already been updated.")
    print("- Missing metadata could affect the score.")

Content ID: content_5feee3994adb
Action: Refresh
Reason Code: STALE_CONTENT
Confidence: Medium
What would make it wrong?
- Traffic may be seasonal.
- Recent Google algorithm changes.
- Content may have already been updated.
- Missing metadata could affect the score.
Content ID: content_b16bd7307b39
Action: Refresh
Reason Code: STALE_CONTENT
Confidence: Medium
What would make it wrong?
- Traffic may be seasonal.
- Recent Google algorithm changes.
- Content may have already been updated.
- Missing metadata could affect the score.
Content ID: content_d34c89fad803
Action: Refresh
Reason Code: STALE_CONTENT
Confidence: Medium
What would make it wrong?
- Traffic may be seasonal.
- Recent Google algorithm changes.
- Content may have already been updated.
- Missing metadata could affect the score.
Content ID: content_ab18b5811c02
Action: Refresh
Reason Code: STALE_CONTENT
Confidence: Medium
What would make it wrong?
- Traffic may be seasonal.
- Recent Google algorithm changes.
- Content may ha

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some lower-ranked pages may have insufficient evidence because of low impressions or missing metadata. These recommendations should be treated as decision-support rather than final decisions.

## Leakage Check

- No future-window information was used.
- No label-derived variables were used.
- `trend_direction` was not used.
- `trend_pct` was not used.
- `content_id` and `client_id` were not used as features.
- The rule relies only on observable search performance signals.

In [42]:
print("Leakage Check")

print("trend_direction used:",
      "trend_direction" in ["days_since_last_update","ctr","impressions_90d","avg_position"])

print("trend_pct used:",
      "trend_pct" in ["days_since_last_update","ctr","impressions_90d","avg_position"])

print("content_id used as feature:",
      False)

print("client_id used as feature:",
      False)

print("No future-window features were intentionally used.")

Leakage Check
trend_direction used: False
trend_pct used: False
content_id used as feature: False
client_id used as feature: False
No future-window features were intentionally used.


In [43]:
features_used = [
    "days_since_last_update",
    "ctr",
    "impressions_90d",
    "avg_position"
]

print(features_used)

['days_since_last_update', 'ctr', 'impressions_90d', 'avg_position']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.